# **Imports and Initialization**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
pip install pyyaml

# **The Loader**

In [58]:
import os
os.makedirs("/content/drive/MyDrive/SISSA_Thesis/src/assembler", exist_ok=True)

In [11]:
%%writefile /content/drive/MyDrive/SISSA_Thesis/src/assembler/jsonl_assembler.py

import json
import glob
import os
import random

def load_all_pairs(pairs_dir: str) -> list[dict]:
    """
    Loads all ch*_review.txt files from pairs_dir and merges
    them into a single flat list of pair dicts.
    """
    all_pairs = []
    pattern = os.path.join(pairs_dir, "ch*_review.txt")
    files = sorted(glob.glob(pattern))

    if not files:
        raise FileNotFoundError(f"No ch*_review.txt files found in {pairs_dir}")

    for fpath in files:
        pairs = parse_review_txt(fpath)
        print(f"  Loaded {len(pairs):>4} pairs from {os.path.basename(fpath)}")
        all_pairs.extend(pairs)

    print(f"\n  Total pairs loaded: {len(all_pairs)}")
    return all_pairs

def parse_review_txt(fpath: str) -> list[dict]:
    """
    Parses a ch*_review.txt file into a list of pair dicts with keys:
    block_id, block_type, instruction, output
    """
    with open(fpath, "r", encoding="utf-8") as f:
        content = f.read()

    # Split on the separator line, drop empty chunks
    separator = "─── Pair"
    chunks = [c.strip() for c in content.split(separator) if c.strip()]

    pairs = []
    for chunk in chunks:
        pair = {}
        lines = chunk.splitlines()

        # First line is "N ───..." — skip it
        lines = lines[1:]

        # We'll reconstruct fields by scanning for known keys
        current_key = None
        current_val_lines = []

        for line in lines:
            if line.startswith("BLOCK:"):
                pair["block_id"] = line[len("BLOCK:"):].strip()
                current_key = None
            elif line.startswith("TYPE:"):
                pair["block_type"] = line[len("TYPE:"):].strip()
                current_key = None
            elif line.startswith("INSTRUCTION:"):
                if current_key == "instruction":
                    pair["instruction"] = " ".join(current_val_lines).strip()
                current_key = "instruction"
                current_val_lines = [line[len("INSTRUCTION:"):].strip()]
            elif line.startswith("OUTPUT:"):
                if current_key == "instruction":
                    pair["instruction"] = " ".join(current_val_lines).strip()
                current_key = "output"
                current_val_lines = [line[len("OUTPUT:"):].strip()]
            else:
                if current_key:
                    current_val_lines.append(line.strip())

        # Flush last field
        if current_key == "output":
            pair["output"] = " ".join(current_val_lines).strip()
        elif current_key == "instruction":
            pair["instruction"] = " ".join(current_val_lines).strip()

        if pair:
            pairs.append(pair)

    return pairs

def filter_pairs(pairs: list[dict]) -> list[dict]:
    """
    Removes pairs with empty outputs.
    """
    before = len(pairs)
    pairs = [p for p in pairs if p.get("output", "").strip()]
    after = len(pairs)
    print(f"  Filtered {before - after} empty-output pairs")
    print(f"  Remaining: {after} pairs")
    return pairs

def deduplicate_pairs(pairs: list[dict]) -> list[dict]:
    """
    Deduplicates pairs on (block_id, instruction).
    Keeps the first occurrence of each unique pair.
    """
    before = len(pairs)
    seen = set()
    deduped = []

    for p in pairs:
        key = (p["block_id"], p["instruction"])
        if key not in seen:
            seen.add(key)
            deduped.append(p)

    after = len(deduped)
    print(f"  Removed {before - after} duplicate pairs")
    print(f"  Remaining: {after} pairs")
    return deduped

def length_filter(pairs: list[dict], max_total_chars: int) -> list[dict]:
    """
    Drops pairs where instruction + output combined exceeds max_total_chars.
    """
    before = len(pairs)
    pairs = [
        p for p in pairs
        if len(p.get("instruction", "")) + len(p.get("output", "")) <= max_total_chars
    ]
    after = len(pairs)
    print(f"  Dropped {before - after} pairs exceeding {max_total_chars} chars")
    print(f"  Remaining: {after} pairs")
    return pairs

def split_pairs(pairs: list[dict], train_ratio: float, seed: int) -> tuple[list[dict], list[dict]]:
    """
    Shuffles pairs with a fixed seed and splits into train/val sets.
    """
    random.seed(seed)
    shuffled = pairs.copy()
    random.shuffle(shuffled)

    split_idx = int(len(shuffled) * train_ratio)
    train = shuffled[:split_idx]
    val = shuffled[split_idx:]

    print(f"  Train: {len(train)} pairs")
    print(f"  Val:   {len(val)} pairs")
    return train, val

def write_jsonl(pairs: list[dict], fpath: str, system_prompt: str) -> None:
    """
    Writes pairs to a JSONL file in ChatML format.
    """
    os.makedirs(os.path.dirname(fpath), exist_ok=True)
    with open(fpath, "w", encoding="utf-8") as f:
        for p in pairs:
            example = {
                "messages": [
                    {"role": "system",    "content": system_prompt},
                    {"role": "user",      "content": p["instruction"]},
                    {"role": "assistant", "content": p["output"]}
                ]
            }
            f.write(json.dumps(example) + "\n")
    print(f"  Wrote {len(pairs)} examples to {os.path.basename(fpath)}")

Overwriting /content/drive/MyDrive/SISSA_Thesis/src/assembler/jsonl_assembler.py


In [12]:
import sys
sys.path.append("/content/drive/MyDrive/SISSA_Thesis/src/assembler")
import importlib
import jsonl_assembler
importlib.reload(jsonl_assembler)
from jsonl_assembler import load_all_pairs, parse_review_txt, filter_pairs, deduplicate_pairs,length_filter, split_pairs, write_jsonl

In [4]:
import os
print(os.listdir("/content/drive/MyDrive/SISSA_Thesis/data/pairs"))

['ch06_review.txt', 'ch07_review.txt', 'ch08_review.txt', 'ch01_review.txt', 'skipped_blocks.log', 'ch02_review.txt', 'ch03_review.txt', 'ch04_review.txt', 'ch05_review.txt']


In [72]:
pairs = parse_review_txt("/content/drive/MyDrive/SISSA_Thesis/data/pairs/ch07_review.txt")
print(f"Parsed {len(pairs)} pairs")
print(pairs[0])   # should have all 4 keys
print(pairs[9])   # pair 10 — output should be empty string ""

Parsed 36 pairs
{'block_id': 'ch7_s1_block_001', 'block_type': 'section_header', 'instruction': 'Summarize the primary thesis presented in this excerpt regarding the study of supersymmetric gauge theories.', 'output': 'The excerpt posits that the various dualities and structures observed in lower-dimensional supersymmetric gauge theories can be better understood by adopting a higher-dimensional framework. This suggests that complex phenomena in 3D and 4D theories are specific manifestations of more general structures residing in higher dimensions.'}
{'block_id': 'ch7_s1_1_block_005', 'block_type': 'equation', 'instruction': 'What does each term in the following equation represent, in the context of SCFTs in Six Dimensions?', 'output': ''}


In [5]:
pairs = load_all_pairs("/content/drive/MyDrive/SISSA_Thesis/data/pairs")

  Loaded  931 pairs from ch01_review.txt
  Loaded  744 pairs from ch02_review.txt
  Loaded  362 pairs from ch03_review.txt
  Loaded  339 pairs from ch04_review.txt
  Loaded  590 pairs from ch05_review.txt
  Loaded   26 pairs from ch06_review.txt
  Loaded   36 pairs from ch07_review.txt
  Loaded   88 pairs from ch08_review.txt

  Total pairs loaded: 3116


In [6]:
pairs = filter_pairs(pairs)

  Filtered 892 empty-output pairs
  Remaining: 2224 pairs


In [7]:
pairs = deduplicate_pairs(pairs)

  Removed 0 duplicate pairs
  Remaining: 2224 pairs


In [8]:
pairs = length_filter(pairs, max_total_chars=3000)

  Dropped 0 pairs exceeding 3000 chars
  Remaining: 2224 pairs


In [9]:
train, val = split_pairs(pairs, train_ratio=0.90, seed=42)

  Train: 2001 pairs
  Val:   223 pairs


In [13]:
SYSTEM_PROMPT = "You are an expert in supersymmetric gauge theories and 3d mirror symmetry. Answer questions accurately and formally, using precise physical and mathematical language."
write_jsonl(train, "/content/drive/MyDrive/SISSA_Thesis/data/final/train.jsonl", SYSTEM_PROMPT)
write_jsonl(val,   "/content/drive/MyDrive/SISSA_Thesis/data/final/val.jsonl",   SYSTEM_PROMPT)

  Wrote 2001 examples to train.jsonl
  Wrote 223 examples to val.jsonl


In [14]:
%%bash
head -n 1 /content/drive/MyDrive/SISSA_Thesis/data/final/train.jsonl | python3 -m json.tool

{
    "messages": [
        {
            "role": "system",
            "content": "You are an expert in supersymmetric gauge theories and 3d mirror symmetry. Answer questions accurately and formally, using precise physical and mathematical language."
        },
        {
            "role": "user",
            "content": "Explain how the structure of the scalar potential in supersymmetric gauge theories leads to the formation of a moduli space of vacua."
        },
        {
            "role": "assistant",
            "content": "A moduli space of vacua emerges in a supersymmetric gauge theory when the scalar potential admits continuous families of zero-energy configurations. Since the potential is constructed such that it is manifestly non-negative, the condition for a vacuum is the simultaneous vanishing of all $F$-terms, determined by the superpotential $\\mathcal{W}$, and all $D$-terms, which depend on the field representation $T^{\\alpha}$ and FI parameters $\\eta^{\\alpha}$. Wh

In [16]:
%%bash
wc -l /content/drive/MyDrive/SISSA_Thesis/data/final/train.jsonl
wc -l /content/drive/MyDrive/SISSA_Thesis/data/final/val.jsonl

2001 /content/drive/MyDrive/SISSA_Thesis/data/final/train.jsonl
223 /content/drive/MyDrive/SISSA_Thesis/data/final/val.jsonl
